In [ ]:
import sys
sys.path.append("..")

from src.data.data_cleaning import load_config
from src.models.train_model import (
    load_processed, split_data, scale_features, train_model, evaluate_model,
    save_model, save_metrics, run_training,
)
import pandas as pd

config = load_config("config/config.yaml")

In [9]:
df = load_processed(config)
df.head()
df.shape

(500, 12)

In [10]:
X_train, X_test, y_train, y_test = split_data(df, config)

In [11]:
X_train_scaled, X_test_scaled, scaler = scale_features(X_train, X_test)

In [12]:
print(pd.DataFrame(X_train_scaled, columns=X_train.columns).describe().loc[["mean", "std"]])

      hours_studied  attendance_rate   sleep_hours  previous_score  \
mean  -1.554312e-17    -1.232348e-15  7.549517e-17    2.664535e-16   
std    1.001252e+00     1.001252e+00  1.001252e+00    1.001252e+00   

      extracurricular        gender  edu_Bachelor  edu_High School  \
mean     6.439294e-17 -2.664535e-17 -7.549517e-17    -8.881784e-18   
std      1.001252e+00  1.001252e+00  1.001252e+00     1.001252e+00   

        edu_Master  study_efficiency  
mean  8.881784e-18     -5.551115e-18  
std   1.001252e+00      1.001252e+00  


In [13]:
model = train_model(X_train_scaled, y_train, config)

In [14]:
import pandas as pd
coef_df = pd.DataFrame({
    "feature": X_train.columns,
    "coefficient": model.coef_[0]
}).sort_values("coefficient", ascending=False)
coef_df


,feature,coefficient
3,previous_score,0.370371
5,gender,0.152177
1,attendance_rate,0.147066
7,edu_High School,0.085997
2,sleep_hours,0.074504
8,edu_Master,0.061237
4,extracurricular,0.008863
0,hours_studied,-0.096978
6,edu_Bachelor,-0.138602
9,study_efficiency,-0.341771


In [15]:
metrics = evaluate_model(model, X_test_scaled, y_test)
metrics

{'accuracy': 0.53,
 'precision': 0.5333333333333333,
 'recall': 0.48,
 'f1_score': 0.5052631578947369}

In [16]:
print(max(X_train["previous_score"]))

98.5


In [17]:
print(model.n_iter_, "dari max_iter:", config["model"]["max_iter"])

[8] dari max_iter: 400


In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

target = config["data"]["target"]
X_all = df.drop(columns=["student_id", target])
y_all = df[target]

pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=config["model"]["max_iter"], random_state=config["model"]["random_state"])),
])

cv_scores = cross_val_score(pipe, X_all, y_all, cv=5, scoring="accuracy")
print("CV scores:", cv_scores)
print("Mean CV accuracy:", cv_scores.mean(), "| Std:", cv_scores.std())

CV scores: [0.44 0.55 0.63 0.55 0.54]
Mean CV accuracy: 0.542 | Std: 0.0604648658313239


In [ ]:
# drop study_efficiency, latih ulang
X_no_derived = df.drop(columns=["student_id", "study_efficiency", config["data"]["target"]])
y = df[config["data"]["target"]]

cv_scores_no_derived = cross_val_score(pipe, X_no_derived, y, cv=5, scoring="accuracy")
print("CV tanpa study_efficiency:", cv_scores_no_derived.mean())

CV tanpa study_efficiency: 0.5439999999999999


In [ ]:
rf = RandomForestClassifier(random_state=42, n_estimators=200)
cv_rf = cross_val_score(rf, X_all, y_all, cv=5, scoring="accuracy")
print("CV Random Forest:", cv_rf.mean())

CV Random Forest: 0.522


In [ ]:
X_no_derived = df.drop(columns=["student_id", "study_efficiency", config["data"]["target"]])
y = df[config["data"]["target"]]

rf = RandomForestClassifier(random_state=42, n_estimators=200)
cv_rf = cross_val_score(rf, X_no_derived, y, cv=5, scoring="accuracy")
print("CV Random Forest:", cv_rf.mean(), "| per fold:", cv_rf)

CV Random Forest: 0.5199999999999999 | per fold: [0.5  0.49 0.54 0.52 0.55]


In [18]:
save_model(model, config)
save_metrics(metrics, config)
print("Model & metrics saved.")

Model & metrics saved.
